In [33]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
nltk.download("punkt", quiet=True)
from nltk.tokenize import sent_tokenize
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments
import torch
from datasets import Dataset, DatasetDict
import evaluate
import re

In [3]:
train = pd.read_csv("C:/Users/User/Desktop/master/proyectos/doc_summ_llm/data/train.csv")
test = pd.read_csv("C:/Users/User/Desktop/master/proyectos/doc_summ_llm/data/test_features.csv")

In [4]:
train.head()

,paper_id,text,summary
0,0,## FROM SOVEREIGNTY TO EXTRATERRITORIAL CONSCI...,"In this article, Victor Fan argues that analys..."
1,1,## 1. Introduction\n\n\nAn Electronic Health R...,Problem definition: Physicians spend more than...
2,2,## Introduction\n\n\nTranslation plays an i...,Literary translation is one of the most challe...
3,3,## 1 Problem Setup\n\n\nRecent political scien...,There is a long-running debate on evaluating f...
4,4,## INTRODUCTION\n\n\nThis article investigat...,"Recently, ‘bimajyo’ (美魔女) came into focus in J..."


In [5]:
test.head()

,paper_id,text
0,1000,## Introduction\n\n\nGender disparities persis...
1,1001,## Introduction\n\n\nOne of humanity’s greates...
2,1002,## Introduction\n\n\nHow do workers get attach...
3,1003,## BETWEEN INDEXES AND SYMBOLS: AN EXPRESSION ...
4,1004,## The Evolution of Environmental and Climate ...


In [6]:
train.shape

(1000, 3)

In [14]:
len(train['summary'][789])

1148

In [50]:
len_total = []
for l in train['summary']:
    len_total.append(len(l))

print(np.mean(len_total))
print(np.std(len_total))

1274.3857142857144
428.4297639672823


In [18]:
validation = train.iloc[700:]
validation.head()

,paper_id,text,summary
700,700,## Introduction\n\n\nCommon sense understandin...,Conventional interpretations of public and soc...
701,701,INTRODUCTION\n\n\nAmerican anthropology is eng...,American Anthropology is engaged in significan...
702,702,## Introduction\n\n\nSwear and taboo words in ...,The current study attempted to determine the t...
703,703,## Introduction\n\n\nAddressing attitudes is c...,Introduction\nAddressing attitudes is central ...
704,704,## Introduction\n\n\nThe concept of time is at...,It is well recognized that time-averaging of a...


In [19]:
train = train.iloc[:700]
train.shape

(700, 3)

In [20]:
validation.shape

(300, 3)

In [21]:
train.drop(columns=['paper_id'], inplace=True)
validation.drop(columns=['paper_id'], inplace=True)
test.drop(columns=['paper_id'], inplace=True)

In [22]:
train.head()

,text,summary
0,## FROM SOVEREIGNTY TO EXTRATERRITORIAL CONSCI...,"In this article, Victor Fan argues that analys..."
1,## 1. Introduction\n\n\nAn Electronic Health R...,Problem definition: Physicians spend more than...
2,## Introduction\n\n\nTranslation plays an i...,Literary translation is one of the most challe...
3,## 1 Problem Setup\n\n\nRecent political scien...,There is a long-running debate on evaluating f...
4,## INTRODUCTION\n\n\nThis article investigat...,"Recently, ‘bimajyo’ (美魔女) came into focus in J..."


In [23]:
train_ds = Dataset.from_pandas(train)
validation_ds = Dataset.from_pandas(validation)
test_ds = Dataset.from_pandas(test)

In [24]:
ds = DatasetDict({
    'train': train_ds,
    'validation': validation_ds,
})

In [25]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'summary'],
        num_rows: 700
    })
    validation: Dataset({
        features: ['text', 'summary'],
        num_rows: 300
    })
})

In [26]:
len(ds['train']['summary'][16])

1030

In [105]:
model_name = 'facebook/bart-large-cnn'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [106]:
model.config

BartConfig {
  "_num_labels": 3,
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "add_final_layer_norm": false,
  "architectures": [
    "BartForConditionalGeneration"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classif_dropout": 0.0,
  "classifier_dropout": 0.0,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 4096,
  "decoder_layerdrop": 0.0,
  "decoder_layers": 12,
  "decoder_start_token_id": 2,
  "dropout": 0.1,
  "dtype": "float32",
  "early_stopping": true,
  "encoder_attention_heads": 16,
  "encoder_ffn_dim": 4096,
  "encoder_layerdrop": 0.0,
  "encoder_layers": 12,
  "eos_token_id": 2,
  "force_bos_token_to_be_generated": true,
  "forced_bos_token_id": 0,
  "forced_eos_token_id": 2,
  "gradient_checkpointing": false,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2"
  },
  "init_std": 0.02,
  "is_encoder_decoder": true,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2
  },
  "lengt

In [87]:
tokenizer.model_max_length
tokenizer.vocab

{'▁Az': 12611,
 '▁Survey': 11418,
 'aţia': 10984,
 '▁contacter': 17987,
 '▁pret': 7140,
 'licate': 26221,
 'kamera': 28867,
 '▁courant': 16203,
 '▁landscape': 3283,
 'verfahren': 16052,
 '▁conclusion': 7489,
 '▁gezeigt': 22985,
 '▁drying': 16773,
 '▁répondre': 13122,
 'schmutz': 31877,
 '<extra_id_52>': 32047,
 '▁Gerne': 25247,
 'Schraub': 31976,
 '▁citi': 16545,
 '512': 24163,
 '▁brochure': 15535,
 '▁aprobat': 22667,
 '▁Natalie': 26983,
 '▁Materialien': 20636,
 'tischen': 13413,
 '▁Schatten': 30632,
 'spring': 14662,
 '▁Beyond': 13594,
 '▁unver': 13705,
 '▁Ty': 10352,
 '▁votre': 618,
 '▁exams': 14026,
 '▁domeniul': 9180,
 '▁unter': 1199,
 '▁traditional': 1435,
 'TIN': 25424,
 'extrait': 28739,
 '▁razor': 26828,
 'REN': 22413,
 'leton': 13248,
 'grilled': 19521,
 'FS': 7674,
 'thankfully': 29706,
 '▁resurse': 18113,
 '▁Hearing': 26651,
 '▁influence': 2860,
 '▁cam': 5511,
 'din': 2644,
 '▁pastel': 22678,
 '▁Luci': 11977,
 '▁route': 2981,
 'leaking': 26177,
 '126': 21976,
 '▁Bäume': 2940

In [ ]:
print(ds['train']['text'][654])

## Introduction


English for Specific Purposes (E.S.P) courses form the foundation of English learning across various academic fields. Munoz-Liza and Taillefer (2018) highlight that (E.S.P) courses address  the  diverse  contextual  needs  of  different  audiences.  Basturkmen  (2006)  notes  that educators implement (E.S.P) in various countries and contexts.

Paltridge  and  Starfield  (2013)  describe  English  for  Academic  Purposes  (E.A.P)  as teaching English to learners who use the language for academic responsibilities. Stojkovi (2015) asserts that English has become the primary language of international communication in scientific and medical fields, linking people across various disciplines. Achievement tests in (E.S.P) courses are crucial for assessing course output and students' proficiency. Hutchinson and Waters (1987) and Hughes (2003) emphasize that (E.S.P) evaluates learners' proficiency and course objectives. According  to  Bachman  (1990),  test  scores  must  have 

In [48]:
print(ds['train']['summary'][654])

Given the critical role of medical English in preparing students for real-world medical environments, this research is vital for ensuring that assessment tools accurately measure the competencies needed for effective communication in medical settings. The study addresses a significant research gap concerning aligning medical English tests with evolving medical language demands and professional practices. This study investigates instructors’ perspectives on the content validity of English for Medical Purposes achievement tests at universities in Saudi Arabia. It seeks answers to how instructors evaluate the alignment, relevance, and validity of English for Medical Purposes achievement test content concerning course objectives, professional language needs, and practical application. The researcher collected data using a structured questionnaire, which he distributed to 32 instructors with significant experience teaching Medical English textbooks. Findings reveal that while most instructo

In [108]:
def preprocess_df(x):
    max_len = 1024
    head_len = 256
    end_len= max_len - head_len

    # Limpias texto
    x['text'] = [re.sub(r'\s+', ' ', str(t)) for t in x['text']]        # elimina espacios repetidos
    x['text'] = [re.sub(r'\n+', ' ', t) for t in x['text']]        # elimina saltos de línea
    x['text'] = [re.sub(r'[^\x00-\x7F]+', ' ', t) for t in x['text']]  # elimina caracteres raros

    # Limpias el resumen 
    x['summary'] = [re.sub(r'\s+', ' ', str(s)) for s in x['summary']]         # elimina espacios repetidos
    x['summary'] = [re.sub(r'\n+', ' ', s) for s in x['summary']]         # elimina saltos de línea
    x['summary'] = [re.sub(r'[^\x00-\x7F]+', ' ', s) for s in x['summary']]  # elimina caracteres raros


    # Tokenizas texto, truncation y padding en el DataCollator
    inputs = tokenizer(x['text'], truncation = False, padding=False)

    # Truncas manualmente
    input_ids_trunc = []
    attention_mask_trunc = []
    for ids in inputs['input_ids']:
        new_ids = ids[:head_len] + ids[-end_len:]
        input_ids_trunc.append(new_ids)
        attention_mask_trunc.append([1] * len(new_ids))

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(x['summary'], max_length=1024, truncation=True, padding=False)

    return ({
        'input_ids': input_ids_trunc,
        'attention_mask': attention_mask_trunc,
        'labels': labels['input_ids']
    })



In [109]:
ds_tokenized = ds.map(preprocess_df, batched=True, batch_size=32)

Map:   0%|          | 0/700 [00:00<?, ? examples/s]c:\Users\User\anaconda3\envs\nlp\lib\site-packages\transformers\tokenization_utils_base.py:4007: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Map: 100%|██████████| 300/300 [00:01<00:00, 160.76 examples/s]


In [91]:
ds_tokenized['train']['labels'][1]

[5289,
 4903,
 10,
 23869,
 7,
 1492,
 72,
 145,
 305,
 716,
 3,
 9,
 239,
 464,
 30,
 9885,
 1685,
 11392,
 41,
 427,
 11120,
 61,
 1002,
 11,
 72,
 145,
 46,
 1781,
 692,
 262,
 11120,
 4145,
 227,
 8,
 414,
 13,
 8,
 161,
 1135,
 5,
 25638,
 302,
 2116,
 43,
 4313,
 8,
 28879,
 1951,
 13,
 10981,
 262,
 11120,
 169,
 11,
 227,
 18,
 5842,
 7,
 161,
 6,
 379,
 10027,
 5958,
 670,
 6,
 10027,
 44,
 1788,
 1575,
 6,
 11,
 4141,
 16735,
 5,
 611,
 6,
 262,
 11120,
 97,
 19,
 59,
 3,
 18760,
 46,
 1215,
 5255,
 1162,
 2945,
 38,
 34,
 5619,
 30,
 10027,
 4742,
 3889,
 24,
 228,
 43,
 359,
 7763,
 7702,
 5,
 3,
 23500,
 6,
 1884,
 6678,
 65,
 59,
 1702,
 48,
 2859,
 17577,
 120,
 5,
 86,
 48,
 1040,
 6,
 62,
 9127,
 149,
 13768,
 31,
 16101,
 3055,
 30,
 116,
 12,
 1912,
 262,
 11120,
 4145,
 2603,
 10,
 5637,
 792,
 97,
 30,
 262,
 11120,
 11,
 6499,
 97,
 1869,
 227,
 161,
 5,
 7717,
 1863,
 87,
 20119,
 7,
 10,
 421,
 331,
 9418,
 300,
 209,
 9286,
 14936,
 45,
 3,
 4581,
 13768,
 45,


In [116]:
training_args = Seq2SeqTrainingArguments(
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    generation_max_length=256,
    fp16=True  
)
data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer, model=model_name, padding=True, return_tensors='pt')

In [117]:
trainer = Seq2SeqTrainer(
    model,
    args = training_args,
    tokenizer = tokenizer,
    train_dataset=ds_tokenized['train'],
    eval_dataset=ds_tokenized['validation'],
    data_collator=data_collator
)

C:\Users\User\AppData\Local\Temp\ipykernel_1832\2483078581.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [118]:
trainer.train()

Step,Training Loss


KeyboardInterrupt: 

In [119]:
from torch.utils.data import DataLoader

def tokenize_test(x):
    max_len = 1024
    head_len = 256
    end_len = max_len - head_len

    x['text'] = [re.sub(r'\s+', ' ', str(t)) for t in x['text']]        # elimina espacios repetidos
    x['text'] = [re.sub(r'\n+', ' ', t) for t in x['text']]        # elimina saltos de línea
    x['text'] = [re.sub(r'[^\x00-\x7F]+', ' ', t) for t in x['text']]  # elimina caracteres raros

    inputs = tokenizer(x['text'], padding = False, truncation=False)

    input_ids_trunc, mask_attention_trunc = [], []
    for ids in inputs['input_ids']:
        new_ids = ids[:head_len] + ids[-end_len:]
        input_ids_trunc.append(new_ids)
        mask_attention_trunc.append([1] * len(new_ids))
    
    return ({
        'input_ids': input_ids_trunc,
        'attention_mask': mask_attention_trunc
    })

In [120]:
test_tokenized = test_ds.map(tokenize_test, batched=True)

Map: 100%|██████████| 345/345 [00:02<00:00, 169.46 examples/s]


In [121]:
test_tokenized = test_tokenized.remove_columns(['text'])

In [122]:
test_tokenized

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 345
})

In [123]:
val_tokenized = validation_ds.map(tokenize_test, batched=True)
val_tokenized = val_tokenized.remove_columns(['text', 'summary'])

Map: 100%|██████████| 300/300 [00:01<00:00, 178.45 examples/s]


In [124]:
val_tokenized

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 300
})

In [125]:
i = 0
ejemplo = val_tokenized[i]
ejemplo

{'input_ids': [0,
  48342,
  24474,
  9732,
  1472,
  1346,
  1033,
  9,
  285,
  2004,
  32,
  36550,
  30,
  10,
  2167,
  333,
  9,
  9553,
  47779,
  3115,
  14,
  33,
  57,
  11359,
  30,
  39061,
  21314,
  4,
  17356,
  4941,
  88,
  285,
  2004,
  19771,
  3489,
  109,
  45,
  14410,
  15,
  5,
  1291,
  227,
  19771,
  8,
  714,
  709,
  6,
  602,
  25,
  4159,
  5,
  592,
  1272,
  14,
  215,
  2004,
  1986,
  2519,
  7,
  4,
  152,
  1566,
  3891,
  6330,
  7,
  4442,
  2210,
  13144,
  30,
  634,
  25328,
  7930,
  579,
  36,
  23301,
  43,
  653,
  579,
  5,
  936,
  4625,
  7,
  28,
  116,
  36,
  771,
  4454,
  43,
  19771,
  1966,
  5448,
  6,
  61,
  16,
  17213,
  11,
  34495,
  438,
  34860,
  811,
  19771,
  1966,
  4,
  1398,
  6,
  305,
  4454,
  16,
  5049,
  7,
  5,
  289,
  5867,
  2981,
  2717,
  16730,
  36,
  13022,
  846,
  238,
  10,
  739,
  2660,
  5057,
  3225,
  1687,
  30,
  5,
  289,
  5867,
  2981,
  1080,
  7,
  42901,
  1233,
  14566,
  9,
  5,
  

In [126]:
loader = DataLoader(
    [ejemplo],
    batch_size=1,
    collate_fn=lambda x: tokenizer.pad(x, return_tensors="pt")
)

for batch in loader:
    batch = {k: v.to(trainer.model.device) for k, v in batch.items() if k in ["input_ids", "attention_mask"]}
    outputs = trainer.model.generate(**batch, max_length=1024)
    decoded = trainer.tokenizer.batch_decode(outputs, skip_special_tokens=True)

You're using a BartTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


In [127]:
print(decoded)

['This article seeks to extend existing literature by using Bacchi s (2009) What s the problem represented to be? (WPR) discourse analysis method, which is rooted in Foucauldian discourse analysis. WPR is used to uncover key rationalisations within the Council s written documents to further understand the construction of said  problems . Here, I provide a brief context of the HDV by reviewing British council housing policy from the election of the 2010 Coalition Government to the present day, tracing the development from the Coalition s  Big Society  programme to the Haringey Development Vehicle. I also review Foucaudian Discourse Analysis, which forms the backbone of my method. After three decades of building decommodified council housing from the post-War era: Displacement and dispossessions in Uppsala, Sweden. (2006). Do urban regeneration programmes improve public health and reduce health inequalities? A synthesis of the evidence from UK policy and practice (1980 2004). Journal of 

In [128]:
print(validation_ds['summary'][i])

Conventional interpretations of public and social housing—embedded in wider neoliberal and capitalist discourses—are not just ‘out there’, existing on their own. In fact, they are constructed by specific forces within and without political discourse. Existing work on public housing policy generally ignores the relationship between policy development and discourse, thereby taking the social problems addressed by such policies as natural and granted. By critically interrogating lodged assumptions, uncovering underlying theories, and peeling back the layers of discourse, this work builds on existing literature by using Bacchi’s What’s the problem represented to be? (WPR) discourse analysis method, which has its roots in Foucauldian discourse analysis. Used in housing studies and other domains, WPR is here applied to the Haringey Development Vehicle (HDV), a £2 billion joint venture considered by the Haringey Council, which was intended to regenerate significant portions of the Borough. WP

In [129]:
trainer.evaluate()

KeyboardInterrupt: 